In [ ]:
import pandas as pd
import shutil
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Data Augmentation

In [ ]:

# Definir el Data Generator con transformaciones
from tensorflow.keras.preprocessing.image import ImageDataGenerator
datagen = ImageDataGenerator(
    rotation_range=20,  
    zoom_range=0.2,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True, 
    fill_mode='nearest',  
)


In [ ]:

save_dir = 'database/datasetV3/train/benigno'  # Especifica el directorio donde se guardarán las imágenes
os.makedirs(save_dir, exist_ok=True)


### Para cada clase se hace el respectivo aumento de datos 

In [ ]:

# Filtrar las imágenes benignas (clase 1)
benign_images = imgs_processor[labels == 1]  # Filtrar las imágenes de la clase 2 (benigno)

# Asegurarse de que las imágenes tienen una dimensión de canal correcta (224, 224, 1)
benign_images = np.expand_dims(benign_images, axis=-1)  # Expande la dimensión del canal

for i, img in enumerate(benign_images):
    img_to_save = (img.squeeze() * 255).astype('uint8')  # Escalar la imagen de [0, 1] a [0, 255]
    filename = f"normal_original_{i}.jpg"  # Nombre de archivo
    filepath = os.path.join(save_dir, filename)
    cv2.imwrite(filepath, img_to_save) 


In [ ]:
if benign_images.shape[0] > 0:
    # Seleccionar la primera imagen y expandir la dimensión del batch
    img = benign_images[0] 
    img = np.expand_dims(img, axis=0)  # Ahora es (1, 224, 224, 1)

    # Crear un generador de augmentación
    aug_iter = datagen.flow(benign_images, batch_size=1, save_to_dir=save_dir, 
                            save_prefix='normal_aug', save_format='jpg')

    # Generar imágenes augmentadas hasta alcanzar 1995
    images_to_generate = 1995 - benign_images.shape[0]  
    generated_images = 0

    while generated_images < images_to_generate:
        # Generar el siguiente lote de imágenes augmentadas
        aug_img_batch = next(aug_iter)
        generated_images += aug_img_batch.shape[0]  # Contar cuántas imágenes se han generado
        
        # Guardar las imágenes augmentadas
        for j in range(aug_img_batch.shape[0]):
            if generated_images >= images_to_generate:
                break  # Si ya se han generado suficientes imágenes, detener el bucle

else:
    print("No se encontraron imágenes con la etiqueta")